# PROJET MODELISATION GEOMETRIQUE

## SUJET

**Surfacique : nuage de points**

De façon similaire au TP sur les maillages, il sera demandé de pouvoir retrouver une tangente et un rayon en chaque point, ainsi qu’un point sur la surface définie par le nuage de points.

In [138]:
# Imports
import numpy as np
import plotly.graph_objects as go
import open3d as o3d

from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from utils import *

# Paramètres
NUM_POINTS = 10000                  # Nombre de points
NUM_NEIGHBORS = 5                   # Nombre de voisin par point
k = 3                               # Pour le calcul de k_near_neighbors
GENERATION_TYPE = 'random_phere'    # Le type de repartition des points

### Générer le nuage de points

In [139]:
def generate_spherical_points(size, num_points, distribution):

    if distribution == 'surface':
        edge = int(np.sqrt(num_points))
        x = np.linspace(-size, size, edge)
        y = np.linspace(-size, size, edge)
        x, y = np.meshgrid(x, y)
        z = x**2 - y**2
        z *= size/np.max(z)
        points = np.column_stack((x.flatten(), y.flatten(), z.flatten()))

    elif distribution == 'regular_phere':
        indices = np.arange(0, num_points, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices/num_points)
        theta = np.pi * (1 + 5**0.5) * indices
        x = size * np.cos(theta) * np.sin(phi)
        y = size * np.sin(theta) * np.sin(phi)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))

    elif distribution == 'random_phere':
        theta = 2 * np.pi * np.random.rand(num_points)
        phi = np.arccos(2 * np.random.rand(num_points) - 1)
        x = size * np.sin(phi) * np.cos(theta)
        y = size * np.sin(phi) * np.sin(theta)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))

    elif distribution == 'pingu':
        fichier_xyz = "models3D/pingu.xyz"
        points = np.loadtxt(fichier_xyz)
        echant = np.linspace(0, points.shape[0] - 1, NUM_POINTS, dtype=int)
        points = points[echant]
        points[:, 1] = -points[:, 1]
        points *= size/np.max(points)

    return points


points = generate_spherical_points(5, NUM_POINTS, GENERATION_TYPE)

### Obtenir les voisins

In [140]:
def find_direct_neighbors(points, num_neighbors):
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, algorithm='auto').fit(points)
    distances, indices = nbrs.kneighbors(points)
    return indices

direct_neighbors = find_direct_neighbors(points, NUM_NEIGHBORS)

### Obtenir les k plus proches voisins

In [141]:
def get_k_near_neighbors(neighbors_indices, k):

    k_near_neighbors = [set() for _ in range(NUM_POINTS)]
    
    for point in range(NUM_POINTS):
        k_near_neighbors[point].add(point)

    for _ in range(k):
        for point in range(NUM_POINTS):
            for neighbor in neighbors_indices[point]:
                k_near_neighbors[point].update(neighbors_indices[neighbor])
    
    for point in range(NUM_POINTS):
        k_near_neighbors[point].remove(point)
    
    return k_near_neighbors


k_near_neighbors = get_k_near_neighbors(direct_neighbors, k=k)

### Calculer les normales sortantes

In [142]:
def get_normal(points, neighbors):

    T1 = []
    T2 = []
    normals = []
    
    for p in range(NUM_POINTS):
        neighbor_points = points[list(neighbors[p])]
        
        pca = PCA(n_components=3)
        pca.fit(neighbor_points)
        
        T1.append(pca.components_[0])
        T2.append(pca.components_[1])

        normal_vector = pca.components_[2]
        #if np.linalg.norm(points[p] + normal_vector) < np.linalg.norm(points[p]) :
        #    normal_vector =  - normal_vector
        normals.append(normal_vector)
    
    return normals, T1, T2


normals, T1, T2 = get_normal(points, k_near_neighbors)

### Trouver les rayons

In [143]:
def get_rayons(points, direct_neighbors):
    rayons = []
    for i, voisins in enumerate(direct_neighbors):
        distances = [np.linalg.norm(points[i] - points[j]) for j in voisins]
        mean_distance = np.mean(distances)
        rayons.append(mean_distance / 2.0)
    return rayons

rayons = get_rayons(points, direct_neighbors)

### Reconstrction de la surface

In [148]:
point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(points)
point_cloud.normals = o3d.utility.Vector3dVector(normals)

# Maillage avec 10 fois plus de points
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(point_cloud, depth=10, width=1, scale=1.0)

vertices = np.asarray(mesh.vertices)
faces = np.asarray(mesh.triangles)

In [149]:
fig = new_fig()

fig.add_trace(go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    color='lightgreen'
))

fig.show()